In [1]:
# PROJECT 3: Delivery Delay & Operations Analytics - Anu GK
# Step 1: Prepare Data
import pandas as pd, numpy as np, plotly.express as px

df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding='latin1')
print("Shape:", df.shape)
print(df[['Delivery Status','Late_delivery_risk','Shipping Mode','Order Region','Days for shipping (real)','Days for shipment (scheduled)']].head())

# Step 2: Data Quality + Create Delay Metrics (Most Important)
df['order_year'] = pd.to_datetime(df['order date (DateOrders)']).dt.year
df['order_month'] = pd.to_datetime(df['order date (DateOrders)']).dt.month
df['Delay_Days'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['Is_Late'] = (df['Late_delivery_risk']==1).astype(int)
df['OnTime'] = (df['Late_delivery_risk']==0).astype(int)

print(f"Data Quality - Duplicates: {df.duplicated().sum()}, Missing: {df.isnull().sum().sum()}")
print(f"On-Time %: {df['OnTime'].mean()*100:.1f}%")
print(f"Avg Delay: {df['Delay_Days'].mean():.1f} days, Late orders: {df['Is_Late'].sum()}")

# Step 3: Segment delays
print("\n--- Step 3: Delay by Warehouse/Carrier ---")
print(df.groupby('Shipping Mode')['Is_Late'].mean().sort_values(ascending=False)*100)
print(df.groupby('Order Region')['Is_Late'].mean().sort_values(ascending=False).head()*100)

# Step 4: High-delay combos (Hotspot)
hotspot = df.groupby(['Shipping Mode','Order Region','Category Name'])['Is_Late'].agg(['mean','count']).reset_index()
hotspot['Late%'] = hotspot['mean']*100
hotspot = hotspot[hotspot['count']>20].sort_values('Late%', ascending=False).head(10)
print("\n--- Step 4: Top 10 Hotspots ---")
print(hotspot)

# Step 5: Simple Risk Model (Decision Tree logic)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

features = ['Days for shipment (scheduled)','Order Item Quantity','Order Item Total','Product Price']
# Encode shipping mode
df_model = pd.get_dummies(df[features + ['Shipping Mode','Is_Late']], columns=['Shipping Mode'], drop_first=True).dropna()
X = df_model.drop('Is_Late', axis=1)
y = df_model['Is_Late']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
print("\n--- Step 5: Delay Risk Model ---")
print(classification_report(y_test, model.predict(X_test)))

# Step 6: Visuals for Dashboard
fig1 = px.histogram(df, x="Shipping Mode", color="Delivery Status", barmode="group", title="Delay by Shipping Mode")
fig2 = px.histogram(df, x="Order Region", color="Is_Late", barmode="group", title="Delay by Region")
fig3 = px.box(df, x="Is_Late", y="Delay_Days", title="Delay Days Distribution")
fig1.show(); fig2.show(); fig3.show()

print("\n--- Step 6: Recommendations Ready ---")
print("Strongest Patterns: Same Day shipping late% highest, Africa/S America high risk, Scheduled days <2 high risk")

Shape: (47560, 53)
    Delivery Status  Late_delivery_risk   Shipping Mode    Order Region  \
0  Advance shipping                   0  Standard Class  Southeast Asia   
1     Late delivery                   1  Standard Class      South Asia   
2  Shipping on time                   0  Standard Class      South Asia   
3  Advance shipping                   0  Standard Class         Oceania   
4  Advance shipping                   0  Standard Class         Oceania   

   Days for shipping (real)  Days for shipment (scheduled)  
0                         3                              4  
1                         5                              4  
2                         4                              4  
3                         3                              4  
4                         2                              4  
Data Quality - Duplicates: 0, Missing: 88849
On-Time %: 41.9%
Avg Delay: 0.6 days, Late orders: 27649

--- Step 3: Delay by Warehouse/Carrier ---
Shipping Mode
Firs


--- Step 6: Recommendations Ready ---
Strongest Patterns: Same Day shipping late% highest, Africa/S America high risk, Scheduled days <2 high risk


In [2]:
df_sample = df.sample(10000, random_state=42)
df_sample.to_csv("DataCo_Sample_10k.csv", index=False)
print("10k sample created - Download this file from Files panel")

10k sample created - Download this file from Files panel
